In [1]:
# Импортируем необходимые библиотеки

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt

# Загружаем Excel-файл с данными
file = 'Данные для тестового задания.xlsx'

# Смотрим названия листов
excel_file = pd.ExcelFile(file)
excel_file.sheet_names

['Данные об аудитории', 'Данные АБ тестов', 'Листеры']

In [2]:
# Загружаем нужные листы

audience = pd.read_excel(file, sheet_name='Данные об аудитории')
ab_tests = pd.read_excel(file, sheet_name='Данные АБ тестов')
listers = pd.read_excel(file, sheet_name='Листеры')

# Посмотрим первые строки таблиц
display(audience.head())
display(ab_tests.head())
display(listers.head())

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


,experiment_num,experiment_group,user_id,revenue
0,1,test,38456,520
1,1,control,13125924,806
2,1,control,9761984,0
3,1,test,11387012,208
4,1,test,18319648,104


,user_id,date,cnt_adverts,age,cnt_contacts,revenue
0,100,2022-01-01,6,21,119,53
1,100,2022-01-02,2,21,200,18
2,100,2022-01-03,6,21,193,42
3,100,2022-01-04,2,21,143,38
4,100,2022-01-05,2,21,190,40


In [3]:
# Вопрос 1
# MAU — количество уникальных пользователей за месяц

mau = audience['user_id'].nunique()

print('MAU:', mau)

MAU: 7639


In [4]:
# Вопрос 2
# Считаем количество уникальных пользователей за каждый день,
# затем находим среднее дневное значение

audience['date'] = pd.to_datetime(audience['date'])

dau_by_day = (
    audience
    .groupby('date')['user_id']
    .nunique()
)

dau = dau_by_day.mean()

print('Средний DAU:', round(dau, 2))
print('Ответ из вариантов:', round(dau))

Средний DAU: 560.47
Ответ из вариантов: 560


In [5]:
# Вопрос 3
# Находим пользователей, которые были в продукте 1 ноября,
# а затем проверяем, сколько из них вернулось 2 ноября

nov_1 = pd.Timestamp('2023-11-01')
nov_2 = pd.Timestamp('2023-11-02')

users_nov_1 = set(
    audience.loc[audience['date'] == nov_1, 'user_id']
)

users_nov_2 = set(
    audience.loc[audience['date'] == nov_2, 'user_id']
)

returned = users_nov_1 & users_nov_2

retention = len(returned) / len(users_nov_1) * 100

print('Пользователей 1 ноября:', len(users_nov_1))
print('Вернулось 2 ноября:', len(returned))
print('Retention Day 1:', round(retention, 1), '%')

Пользователей 1 ноября: 623
Вернулось 2 ноября: 166
Retention Day 1: 26.6 %


In [6]:
# Вопрос 4
# Анализ retention-кривых двух продуктов

answer_4 = """
Голубой продукт удерживает пользователей значительно лучше.
После первоначального снижения его retention постепенно стабилизируется
примерно на уровне 40%.

У красного продукта retention продолжает быстро снижаться
и примерно к 5-му дню достигает 0%.

Вывод: у голубого продукта значительно лучше удержание пользователей.
"""

print(answer_4)


Голубой продукт удерживает пользователей значительно лучше.
После первоначального снижения его retention постепенно стабилизируется
примерно на уровне 40%.

У красного продукта retention продолжает быстро снижаться
и примерно к 5-му дню достигает 0%.

Вывод: у голубого продукта значительно лучше удержание пользователей.



In [7]:
# Вопрос 5
# Сначала считаем суммарное количество просмотренных объявлений
# для каждого пользователя

adverts_by_user = (
    audience
    .groupby('user_id')['view_adverts']
    .sum()
)

# Пользователь считается сконвертировавшимся,
# если посмотрел хотя бы одно объявление

users_with_adverts = (adverts_by_user > 0).sum()
total_users = adverts_by_user.size

conversion = users_with_adverts / total_users * 100

print('Всего пользователей:', total_users)
print('Посмотрели хотя бы одно объявление:', users_with_adverts)
print('Конверсия:', round(conversion, 1), '%')

Всего пользователей: 7639
Посмотрели хотя бы одно объявление: 3538
Конверсия: 46.3 %


In [8]:
# Вопрос 6
# Считаем среднее количество просмотренных объявлений
# на одного уникального пользователя

avg_adverts = adverts_by_user.mean()

print(
    'Среднее количество объявлений на пользователя:',
    round(avg_adverts, 1)
)

Среднее количество объявлений на пользователя: 2.9


In [9]:
# Вопрос 7
# NPS = % сторонников - % критиков

total_users = 2000
promoters = 1200
detractors = 500

promoters_percent = promoters / total_users * 100
detractors_percent = detractors / total_users * 100

nps = promoters_percent - detractors_percent

print('Сторонники:', promoters_percent, '%')
print('Критики:', detractors_percent, '%')
print('NPS:', nps, '%')

Сторонники: 60.0 %
Критики: 25.0 %
NPS: 35.0 %


In [10]:
# Вопрос 8
# Анализируем три независимых A/B-теста.
# Для каждого эксперимента сравниваем ARPU control и test
# с помощью t-теста.

for experiment in sorted(ab_tests['experiment_num'].unique()):

    exp_data = ab_tests[
        ab_tests['experiment_num'] == experiment
    ]

    control = exp_data[
        exp_data['experiment_group'] == 'control'
    ]['revenue']

    test = exp_data[
        exp_data['experiment_group'] == 'test'
    ]['revenue']

    arpu_control = control.mean()
    arpu_test = test.mean()

    # Welch t-test — не предполагает одинаковую дисперсию групп
    t_stat, p_value = stats.ttest_ind(
        control,
        test,
        equal_var=False
    )

    print('Эксперимент:', experiment)
    print('ARPU control:', round(arpu_control, 2))
    print('ARPU test:', round(arpu_test, 2))
    print('p-value:', round(p_value, 6))

    if p_value < 0.05:
        print('Разница статистически значима.')
    else:
        print('Статистически значимой разницы нет.')

    print('-' * 40)

Эксперимент: 1
ARPU control: 722.46
ARPU test: 665.74
p-value: 0.688966
Статистически значимой разницы нет.
----------------------------------------
Эксперимент: 2
ARPU control: 704.65
ARPU test: 332.93
p-value: 0.001128
Разница статистически значима.
----------------------------------------
Эксперимент: 3
ARPU control: 663.21
ARPU test: 998.67
p-value: 0.060315
Статистически значимой разницы нет.
----------------------------------------


In [12]:
# Выводы по вопросу 8

print("""
Эксперимент 1:
Статистически значимой разницы между группами нет.
Внедрять тестовый вариант на основании этого эксперимента не стоит.

Эксперимент 2:
Разница статистически значима.
ARPU тестовой группы значительно ниже контрольной.
Тестовый вариант внедрять не рекомендуется.

Эксперимент 3:
ARPU тестовой группы выше, однако p-value больше 0.05.
Статистической значимости на уровне 5% нет.
Рекомендуется продолжить эксперимент или увеличить выборку.
""")


Эксперимент 1:
Статистически значимой разницы между группами нет.
Внедрять тестовый вариант на основании этого эксперимента не стоит.

Эксперимент 2:
Разница статистически значима.
ARPU тестовой группы значительно ниже контрольной.
Тестовый вариант внедрять не рекомендуется.

Эксперимент 3:
ARPU тестовой группы выше, однако p-value больше 0.05.
Статистической значимости на уровне 5% нет.
Рекомендуется продолжить эксперимент или увеличить выборку.



In [13]:
# Вопрос 9
# Сначала суммируем доход каждого пользователя,
# затем находим средний доход среди уникальных пользователей

revenue_by_user = (
    listers
    .groupby('user_id')['revenue']
    .sum()
)

avg_revenue = revenue_by_user.mean()

print(
    'Средний доход на пользователя:',
    round(avg_revenue, 1)
)

Средний доход на пользователя: 156.5


In [14]:
# Вопрос 10
# Один пользователь может встречаться несколько раз,
# поэтому сначала оставляем один возраст для каждого пользователя

age_by_user = (
    listers
    .groupby('user_id')['age']
    .first()
)

median_age = age_by_user.median()

print('Медиана возраста:', median_age)

Медиана возраста: 28.0


In [15]:
# Вопрос 11
# Лучший тип графика для сравнения разброса цен
# в разных магазинах — box plot (ящик с усами).

answer_11 = 'Ящик с усами (box plot)'

print('Ответ:', answer_11)

Ответ: Ящик с усами (box plot)


In [16]:
# Box plot показывает:
# - медиану;
# - нижний и верхний квартили;
# - разброс значений;
# - возможные выбросы.

In [17]:
# Вопрос 12
# Бимодальное распределение имеет два выраженных пика.
# На изображениях в задании это график №3.

answer_12 = 'График №3'

print('Ответ:', answer_12)

Ответ: График №3


In [18]:
# Вопрос 13
# Чем сильнее значения распределены относительно среднего,
# тем больше дисперсия.
# Наибольший разброс среди представленных графиков имеет №3.

answer_13 = 'График №3'

print('Ответ:', answer_13)

Ответ: График №3


In [19]:
# Вопрос 14
# Для анализа взаимосвязи двух количественных переменных
# подходит диаграмма рассеяния.
# Корреляционная тепловая карта также показывает
# рассчитанные коэффициенты корреляции.

answer_14 = [
    'Scatter Plot (диаграмма рассеяния)',
    'Correlation Heatmap'
]

print('Подходящие графики:')
for answer in answer_14:
    print('-', answer)

Подходящие графики:
- Scatter Plot (диаграмма рассеяния)
- Correlation Heatmap


In [20]:
# Вопрос 15
# p-value = 0.05 означает:
# если нулевая гипотеза верна,
# вероятность получить такой или ещё более экстремальный
# результат составляет 5%.

answer_15 = """
Есть 5% вероятность случайно получить такой или еще более
экстремальный результат, если нулевая гипотеза верна.
"""

print(answer_15)


Есть 5% вероятность случайно получить такой или еще более
экстремальный результат, если нулевая гипотеза верна.



In [21]:
# Вопрос 16
# Для сравнения средних двух выборок из нормального
# распределения используется t-тест.

answer_16 = 't-тест'

print('Ответ:', answer_16)

Ответ: t-тест


In [22]:
# Вопрос 17
# Квартили делят упорядоченный набор данных
# на четыре части.

answer_17 = 'Квартили делят данные на четыре равные части'

print('Ответ:', answer_17)

Ответ: Квартили делят данные на четыре равные части


In [23]:
# Вопрос 18
# Данные эксперимента

visitors_a = 100_047_501
payments_a = 1003

visitors_b = 100_001_055
payments_b = 1099

# Рассчитываем конверсии

conversion_a = payments_a / visitors_a
conversion_b = payments_b / visitors_b

print(
    'Конверсия A:',
    round(conversion_a * 100, 6),
    '%'
)

print(
    'Конверсия B:',
    round(conversion_b * 100, 6),
    '%'
)

# Считаем относительный прирост

uplift = (
    conversion_b / conversion_a - 1
) * 100

print(
    'Относительный прирост B:',
    round(uplift, 2),
    '%'
)

Конверсия A: 0.001003 %
Конверсия B: 0.001099 %
Относительный прирост B: 9.62 %


In [24]:
# Проверяем статистическую значимость разницы конверсий
# с помощью z-теста для двух долей

pooled_conversion = (
    payments_a + payments_b
) / (
    visitors_a + visitors_b
)

standard_error = np.sqrt(
    pooled_conversion
    * (1 - pooled_conversion)
    * (
        1 / visitors_a
        + 1 / visitors_b
    )
)

z_stat = (
    conversion_b - conversion_a
) / standard_error

p_value = 2 * (
    1 - norm.cdf(abs(z_stat))
)

print('z-statistic:', round(z_stat, 4))
print('p-value:', round(p_value, 4))

z-statistic: 2.1046
p-value: 0.0353


In [25]:
# Вывод по вопросу 18

alpha = 0.05

if p_value < alpha:
    print("""
Разница между группами статистически значима
при уровне значимости 5%.

Вариант B показывает более высокую конверсию.
Относительный прирост составляет примерно 9.6%.

Можно рекомендовать вариант B,
при этом дополнительно следует оценить
экономическую значимость результата.
""")
else:
    print("""
Статистически значимой разницы между вариантами
A и B не обнаружено.
""")


Разница между группами статистически значима
при уровне значимости 5%.

Вариант B показывает более высокую конверсию.
Относительный прирост составляет примерно 9.6%.

Можно рекомендовать вариант B,
при этом дополнительно следует оценить
экономическую значимость результата.

